In [1]:
# Test that all our packages imported correctly
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import plotly.graph_objects as go

print("All packages loaded successfully")

Matplotlib is building the font cache; this may take a moment.


All packages loaded successfully


In [3]:
# Verify yfinance works with a live ticker
test = yf.Ticker("MSFT")
print(test.info.get('longName'))
print(test.info.get('marketCap'))

Microsoft Corporation
3059363872768


In [5]:
# Test Salesforce data availability
crm = yf.Ticker("CRM")

print(crm.info.get('longName'))
print(f"Sector: {crm.info.get('sector')}")
print(f"Industry: {crm.info.get('industry')}")
print(f"Market Cap: ${crm.info.get('marketCap'):,}")
print(f"Current Price: ${crm.info.get('currentPrice')}")

Salesforce, Inc.
Sector: Technology
Industry: Software - Application
Market Cap: $150,089,940,992
Current Price: $183.26


In [6]:
# Check what financial statements are available
income_stmt = crm.financials
cashflow = crm.cashflow
balance_sheet = crm.balance_sheet

print("Income Statement shape:", income_stmt.shape)
print("Cash Flow shape:", cashflow.shape)
print("Balance Sheet shape:", balance_sheet.shape)

print("\nIncome Statement columns (years available):")
print(income_stmt.columns.tolist())

Income Statement shape: (47, 5)
Cash Flow shape: (52, 5)
Balance Sheet shape: (68, 5)

Income Statement columns (years available):
[Timestamp('2026-01-31 00:00:00'), Timestamp('2025-01-31 00:00:00'), Timestamp('2024-01-31 00:00:00'), Timestamp('2023-01-31 00:00:00'), Timestamp('2022-01-31 00:00:00')]


In [7]:
# Print all available income statement line items
print("Available Income Statement items:")
print(income_stmt.index.tolist())

Available Income Statement items:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Normalized EBITDA', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Reconciled Cost Of Revenue', 'EBITDA', 'EBIT', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Total Expenses', 'Total Operating Income As Reported', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Income Expense', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Write Off', 'Restructuring And Mergern Acquisition', 'Gain On Sale Of Security', 'Operating Income', 'Operating Expense', 'Depreciation Amortization Depletion Income Statement', 'De

In [8]:
# Extract the key income statement items we need
# .loc pulls a specific row by label
# .sort_index() sorts columns chronologically (oldest to newest)

income_clean = income_stmt.loc[
    ['Total Revenue', 'EBIT', 'Tax Rate For Calcs', 
     'Reconciled Depreciation', 'Tax Provision']
].sort_index(axis=1)

# Divide by 1,000,000 to convert from dollars to millions
income_clean = income_clean / 1_000_000

# Round to 1 decimal place for readability
income_clean = income_clean.round(1)

# Rename columns to just the fiscal year for cleanliness
income_clean.columns = [col.year for col in income_clean.columns]

print(income_clean)

                         2022     2023     2024     2025     2026
Total Revenue             NaN  31352.0  34857.0  37895.0  41525.0
EBIT                      NaN   1858.0   5999.0   7666.0   8917.0
Tax Rate For Calcs        NaN      0.0      0.0      0.0      0.0
Reconciled Depreciation   NaN   3786.0   3959.0   3477.0   3631.0
Tax Provision             NaN    452.0    814.0   1241.0   2063.0


In [9]:
# Drop 2022 since it's missing data
income_clean = income_clean.drop(columns=[2022])

# Calculate effective tax rate manually
# Tax Rate = Tax Provision / Pretax Income
pretax = income_stmt.loc['Pretax Income'].sort_index() / 1_000_000
tax = income_clean.loc['Tax Provision']

# Align the years
pretax.index = [idx.year for idx in pretax.index]
pretax = pretax.drop(index=2022)

tax_rate = (tax / pretax).round(3)

print("Effective Tax Rate by year:")
print(tax_rate)

print("\nCleaned Income Statement:")
print(income_clean)

Effective Tax Rate by year:
2023    0.685
2024    0.164
2025    0.167
2026    0.217
dtype: float64

Cleaned Income Statement:
                            2023     2024     2025     2026
Total Revenue            31352.0  34857.0  37895.0  41525.0
EBIT                      1858.0   5999.0   7666.0   8917.0
Tax Rate For Calcs           0.0      0.0      0.0      0.0
Reconciled Depreciation   3786.0   3959.0   3477.0   3631.0
Tax Provision              452.0    814.0   1241.0   2063.0


In [10]:
# 2023 tax rate is distorted by one-time restructuring charges
# Use average of 2024-2026 as our normalised rate
normalised_tax_rate = tax_rate[[2024, 2025, 2026]].mean().round(3)

print(f"Distorted 2023 tax rate: {tax_rate[2023]:.1%}")
print(f"Normalised tax rate (2024-2026 avg): {normalised_tax_rate:.1%}")


Distorted 2023 tax rate: 68.5%
Normalised tax rate (2024-2026 avg): 18.3%


In [11]:
# Inspect available cash flow items
print("Available Cash Flow items:")
print(cashflow.index.tolist())

Available Cash Flow items:
['Free Cash Flow', 'Repurchase Of Capital Stock', 'Repayment Of Debt', 'Issuance Of Debt', 'Capital Expenditure', 'Interest Paid Supplemental Data', 'Income Tax Paid Supplemental Data', 'End Cash Position', 'Beginning Cash Position', 'Effect Of Exchange Rate Changes', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Proceeds From Stock Option Exercised', 'Cash Dividends Paid', 'Common Stock Dividend Paid', 'Net Common Stock Issuance', 'Common Stock Payments', 'Net Issuance Payments Of Debt', 'Net Long Term Debt Issuance', 'Long Term Debt Payments', 'Long Term Debt Issuance', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net Business Purchase And Sale', 'Purchase Of Business', 'Capital Expenditure Reported', 'Operating Cash Flow', 'Cash Flow From Continuing Operating Activitie

In [12]:
# Extract cash flow items we need
cf_items = cashflow.loc[
    ['Capital Expenditure', 'Change In Working Capital', 
     'Operating Cash Flow', 'Free Cash Flow']
].sort_index(axis=1)

# Convert to millions and round
cf_clean = (cf_items / 1_000_000).round(1)

# Rename columns to fiscal year
cf_clean.columns = [col.year for col in cf_clean.columns]

# Drop 2022 to match our income statement
cf_clean = cf_clean.drop(columns=[2022])

print(cf_clean)

                             2023     2024     2025     2026
Capital Expenditure        -798.0   -736.0   -658.0   -594.0
Change In Working Capital -2069.0  -2850.0  -1981.0   -781.0
Operating Cash Flow        7111.0  10234.0  13092.0  14996.0
Free Cash Flow             6313.0   9498.0  12434.0  14402.0


In [13]:
# Calculate FCF manually using our formula:
# FCF = EBIT x (1 - TaxRate) + D&A - Capex - Change in Working Capital

ebit = income_clean.loc['EBIT']
da = income_clean.loc['Reconciled Depreciation']
capex = cf_clean.loc['Capital Expenditure']
change_wc = cf_clean.loc['Change In Working Capital']

# Apply normalised tax rate to all years
nopat = ebit * (1 - normalised_tax_rate)  # Net Operating Profit After Tax

fcf_manual = nopat + da + capex - change_wc

print("Our FCF calculation:")
print(fcf_manual.round(1))

print("\nyfinance FCF (cross-check):")
print(cf_clean.loc['Free Cash Flow'])

print("\nDifference:")
print((fcf_manual - cf_clean.loc['Free Cash Flow']).round(1))

Our FCF calculation:
2023     6575.0
2024    10974.2
2025    11063.1
2026    11103.2
dtype: float64

yfinance FCF (cross-check):
2023     6313.0
2024     9498.0
2025    12434.0
2026    14402.0
Name: Free Cash Flow, dtype: float64

Difference:
2023     262.0
2024    1476.2
2025   -1370.9
2026   -3298.8
dtype: float64


In [14]:
# Revised FCF using actual tax provision instead of normalised rate
# NOPAT = EBIT - actual taxes paid
actual_tax = income_clean.loc['Tax Provision']
nopat_actual = ebit - actual_tax

fcf_revised = nopat_actual + da + capex - change_wc

print("Revised FCF (actual taxes):")
print(fcf_revised.round(1))

print("\nyfinance FCF (cross-check):")
print(cf_clean.loc['Free Cash Flow'])

print("\nDifference:")
print((fcf_revised - cf_clean.loc['Free Cash Flow']).round(1))

Revised FCF (actual taxes):
2023     6463.0
2024    11258.0
2025    11225.0
2026    10672.0
dtype: float64

yfinance FCF (cross-check):
2023     6313.0
2024     9498.0
2025    12434.0
2026    14402.0
Name: Free Cash Flow, dtype: float64

Difference:
2023     150.0
2024    1760.0
2025   -1209.0
2026   -3730.0
dtype: float64


In [15]:
# Final FCF calculation using bottom-up approach
# FCF = Operating Cash Flow - Capex
# Note: Capex is already negative, so we add it

operating_cf = cf_clean.loc['Operating Cash Flow']
capex = cf_clean.loc['Capital Expenditure']

fcf_final = operating_cf + capex

print("Final FCF (Operating CF - Capex):")
print(fcf_final.round(1))

print("\nyfinance FCF (cross-check):")
print(cf_clean.loc['Free Cash Flow'])

print("\nDifference (should be near zero):")
print((fcf_final - cf_clean.loc['Free Cash Flow']).round(1))

Final FCF (Operating CF - Capex):
2023     6313.0
2024     9498.0
2025    12434.0
2026    14402.0
dtype: float64

yfinance FCF (cross-check):
2023     6313.0
2024     9498.0
2025    12434.0
2026    14402.0
Name: Free Cash Flow, dtype: float64

Difference (should be near zero):
2023    0.0
2024    0.0
2025    0.0
2026    0.0
dtype: float64


In [16]:
print("Data acquisition complete")
print(f"Historical FCF (2023-2026): {fcf_final.tolist()}")
print(f"Normalised tax rate: {normalised_tax_rate:.1%}")
print(f"Years of data: {list(fcf_final.index)}")


Data acquisition complete
Historical FCF (2023-2026): [6313.0, 9498.0, 12434.0, 14402.0]
Normalised tax rate: 18.3%
Years of data: [2023, 2024, 2025, 2026]
